# Taller RAG: LangChain + OpenAI + Pinecone

**Objetivo:** construir dos variantes del mismo RAG: recuperación local en memoria y recuperación remota mediante Pinecone.

**Arquitectura:** `Pregunta → embedding → búsqueda semántica → fragmentos recuperados → prompt con contexto → OpenAI → respuesta con fuentes`.

**Parte 1:** `InMemoryVectorStore` mantiene embeddings en memoria. **Parte 2:** `PineconeVectorStore` persiste embeddings en un índice remoto. Se reutilizan el mismo corpus, el mismo modelo de embeddings y el mismo prompt para una comparación controlada.

> Requiere claves reales de OpenAI y Pinecone. No ejecutar las celdas de integración sin configurarlas.

## 0. Preparación y configuración segura

In [ ]:
import os, time, uuid
from pathlib import Path
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

load_dotenv()
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = ROOT / "data" / "knowledge_base.txt"
assert DATA_PATH.exists(), f"No se encontró {DATA_PATH}. Abre Jupyter desde la raíz del repositorio."
assert os.getenv("OPENAI_API_KEY"), "Configura OPENAI_API_KEY en .env"
chat_model = os.getenv("OPENAI_CHAT_MODEL", "gpt-4o-mini")
embedding_model = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
llm = ChatOpenAI(model=chat_model, temperature=0)
embeddings = OpenAIEmbeddings(model=embedding_model)
print("Configuración de OpenAI cargada; claves no mostradas.")

## 1. Base de conocimiento y segmentación

Se divide el corpus en fragmentos con solapamiento para mejorar la recuperación y conservar el contexto. Cada fragmento incluye metadatos de procedencia.

In [ ]:
text = DATA_PATH.read_text(encoding="utf-8")
source_document = Document(page_content=text, metadata={"source": "data/knowledge_base.txt"})
splitter = RecursiveCharacterTextSplitter(chunk_size=350, chunk_overlap=60)
docs = splitter.split_documents([source_document])
for i, doc in enumerate(docs):
    doc.metadata["chunk_id"] = i
print(f"Documentos originales: 1 | Fragmentos: {len(docs)}")
assert docs

## 2. Cadena RAG reutilizable

El prompt restringe las respuestas al contexto recuperado y exige reconocer cuándo falta evidencia. Se muestran los fragmentos utilizados para facilitar la trazabilidad.

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Responde únicamente con el contexto suministrado. Si la respuesta no está en el contexto, di: 'No hay información suficiente en la base de conocimiento'. Responde en español y de forma concisa. Contexto:\n{context}"),
    ("human", "{question}"),
])

def ask_rag(vectorstore, question, k=3):
    start = time.perf_counter()
    retrieved = vectorstore.similarity_search(question, k=k)
    context = "\n\n".join(f"[Fuente {i+1}: {d.metadata.get('source')} / fragmento {d.metadata.get('chunk_id')}] {d.page_content}" for i,d in enumerate(retrieved))
    response = llm.invoke(prompt.invoke({"context": context, "question": question}))
    return {"question": question, "answer": response.content, "sources": [d.metadata for d in retrieved], "fragments": [d.page_content for d in retrieved], "latency_seconds": round(time.perf_counter()-start, 3)}

## Parte 1. RAG con LangChain y OpenAI (almacenamiento local en memoria)

In [ ]:
local_store = InMemoryVectorStore.from_documents(docs, embedding=embeddings)
questions = [
    "¿Cómo evita Campus Café que se creen pedidos duplicados?",
    "¿Qué servicios AWS utiliza Campus Café?",
    "¿Cuál es el horario de atención de Campus Café?",
]
local_results = [ask_rag(local_store, q) for q in questions]
for result in local_results:
    print("\nPREGUNTA:", result["question"])
    print("RESPUESTA:", result["answer"])
    print("FUENTES:", result["sources"])
    print("LATENCIA:", result["latency_seconds"], "s")

## Parte 2. RAG con Pinecone

**Cambio arquitectónico:** sustituimos `InMemoryVectorStore` por `PineconeVectorStore`. Pinecone mantiene los vectores entre sesiones y facilita su búsqueda desde distintas aplicaciones. Se crea un índice de dimensión 1536, compatible con `text-embedding-3-small` por defecto. Si cambias el modelo de embeddings, verifica la dimensión.

**Precaución:** crear un índice o ejecutar consultas puede generar cargos según el plan de cada proveedor.

In [ ]:
assert os.getenv("PINECONE_API_KEY"), "Configura PINECONE_API_KEY en .env"
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

index_name = os.getenv("PINECONE_INDEX_NAME", "campus-cafe-rag").lower()
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
existing = set(pc.list_indexes().names())
if index_name not in existing:
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    print("Índice creado:", index_name)
else:
    print("Índice existente:", index_name)

for attempt in range(30):
    if pc.describe_index(index_name).status.get("ready", False):
        break
    time.sleep(2)
else:
    raise TimeoutError("El índice Pinecone no estuvo listo a tiempo")

# Namespace único evita mezclar datos con pruebas anteriores.
namespace = "workshop-" + uuid.uuid4().hex[:10]
pinecone_store = PineconeVectorStore(
    index_name=index_name, embedding=embeddings, namespace=namespace
)
pinecone_store.add_documents(docs, ids=[f"chunk-{i}" for i in range(len(docs))])
print("Fragmentos cargados:", len(docs), "| Namespace:", namespace)

In [ ]:
pinecone_results = [ask_rag(pinecone_store, q) for q in questions]
for result in pinecone_results:
    print("\nPREGUNTA:", result["question"])
    print("RESPUESTA:", result["answer"])
    print("FUENTES:", result["sources"])
    print("LATENCIA:", result["latency_seconds"], "s")

## 3. Comparación de resultados

La latencia es **orientativa**, no un benchmark reproducible: depende de la red, caché, servicio OpenAI, cantidad de fragmentos y condiciones del índice. La comparación no mide calidad automáticamente; se recomienda revisar manualmente relevancia y fundamentación de las respuestas.

In [ ]:
print(f"{'Pregunta':50} {'Local(s)':>10} {'Pinecone(s)':>12}")
for a,b in zip(local_results,pinecone_results):
    print(f"{a['question'][:50]:50} {a['latency_seconds']:10.3f} {b['latency_seconds']:12.3f}")
print("\nLocal: almacenamiento efímero, simple para prototipos.")
print("Pinecone: persistencia y servicio vectorial gestionado, con latencia y costos de red.")

## 4. Conclusiones y límites

- **LangChain** orquesta la carga, fragmentación, recuperación y generación.
- **OpenAI Embeddings** transforma texto y consultas en vectores.
- **Pinecone** permite persistir y consultar embeddings de forma remota; no sustituye el LLM.
- La recuperación semántica puede fallar: mostrar fuentes y restringir el prompt reduce, pero no elimina, respuestas no fundamentadas.
- Para producción: gestionar secretos con un gestor de secretos, añadir controles de acceso al corpus, métricas de calidad, costos, observabilidad y política de retención de datos.

**Evidencia para la entrega:** ejecutar todas las celdas con claves propias, conservar las salidas del notebook y subir el repositorio sin `.env`.